# FracAtlas tuning and final model selection

Run this notebook once. It tunes all three architectures, selects the best trial for each, compares the winners by fracture-focused average recall, evaluates the winner on the untouched test set, and saves the final model for FastAPI.

In [1]:
from pathlib import Path
import itertools, json, math, sys, gc
SERVICE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(SERVICE_DIR))
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight
from app.config import ARTIFACT_DIR, IMAGE_SIZE, SEED
from app.data import load_manifest, split_manifest, undersample_majority
from app.labels import CLASS_NAMES
from app.metrics import FRACTURE_CLASS_INDICES, SELECTION_METRIC
from app.model import MODEL_NAMES, build_model, build_transfer_model


tf.keras.utils.set_random_seed(SEED)
EPOCHS = 5
MAX_TRIALS_PER_MODEL = 2
MODEL_DIR = ARTIFACT_DIR / 'tuning'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

I0000 00:00:1787800425.816222     380 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1787800426.302169     380 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1787800428.702763     380 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
manifest = load_manifest()
train, validation, test = split_manifest(manifest)
print(f'Images: {len(manifest)} | train: {len(train)} | validation: {len(validation)} | test: {len(test)}')

def make_dataset(frame, batch_size, shuffle):
    paths = frame['path'].to_numpy()
    labels = frame['label_index'].to_numpy(dtype=np.int32)
    def load(path, label):
        image = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
        return tf.image.resize(image, IMAGE_SIZE), label
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle: dataset = dataset.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    return dataset.map(load, num_parallel_calls=1).apply(tf.data.experimental.ignore_errors()).batch(batch_size).prefetch(1)

def calculate_metrics(actual, predicted):
    labels = np.arange(len(CLASS_NAMES))
    p, r, f, _ = precision_recall_fscore_support(actual, predicted, labels=labels, average='macro', zero_division=0)
    fp, fr, ff, _ = precision_recall_fscore_support(actual, predicted, labels=FRACTURE_CLASS_INDICES, average='macro', zero_division=0)
    return {'accuracy': float(accuracy_score(actual, predicted)), 'balanced_accuracy': float(balanced_accuracy_score(actual, predicted)), 'macro_precision': float(p), 'macro_recall': float(r), 'macro_f1': float(f), 'fracture_macro_precision': float(fp), 'fracture_macro_recall': float(fr), 'fracture_macro_f1': float(ff)}

def is_better(candidate, current):
    return current is None or tuple(candidate[key] for key in (SELECTION_METRIC, 'macro_f1', 'balanced_accuracy')) > tuple(current[key] for key in (SELECTION_METRIC, 'macro_f1', 'balanced_accuracy'))

W0000 00:00:1787800458.054379     380 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
W0000 00:00:1787800458.056352     380 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
I0000 00:00:1787800458.293830     380 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5233 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 12.0a
E0000 00:00:1787800491.340749     380 jpeg_mem.cc:331] Premature end of JPEG data. Stopped at line 414/454
W0000 00:00:1787800491.340808     380 local_rendezvous.cc:412] Local rendezvous is aborting with status: INVALID_ARGUMENT: jpeg::Uncompress failed. Invalid JPEG data or crop win

Images: 4024 | train: 3219 | validation: 402 | test: 403


/tmp/ipykernel_380/1627999239.py:1: RuntimeWarning: Skipped 59 unreadable image file(s) from the manifest.
  manifest = load_manifest()


In [3]:
search_space = list(itertools.product((1e-3, 3e-4), (16, 32), (False, True)))

if MAX_TRIALS_PER_MODEL > 0: search_space = search_space[:MAX_TRIALS_PER_MODEL]

all_results, best_by_model = [], {}

for model_name in MODEL_NAMES:
    best = None
    best_path = MODEL_DIR / f'{model_name}_best.keras'

    for trial, (learning_rate, batch_size, undersample) in enumerate(search_space, start=1):
        print(f'===== Training {model_name} trial {trial}/{len(search_space)} =====', flush=True)

        tf.keras.backend.clear_session()
        trial_train = undersample_majority(train) if undersample else train
        weights = compute_class_weight(class_weight='balanced', classes=np.arange(len(CLASS_NAMES)), y=trial_train['label_index'].to_numpy())

        class_weights = {index: float(value) for index, value in enumerate(weights)}
        model = build_model(learning_rate=learning_rate) if model_name == 'custom_cnn' else build_transfer_model(model_name, learning_rate=learning_rate)

        model.fit(make_dataset(trial_train, batch_size, True).repeat(), validation_data=make_dataset(validation, batch_size, False).repeat(), epochs=EPOCHS, steps_per_epoch=math.ceil(len(trial_train)/batch_size), validation_steps=math.ceil(len(validation)/batch_size), class_weight=class_weights, callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)], verbose=1)

        probabilities = model.predict(make_dataset(validation, batch_size, False), verbose=0)

        row = {'model': model_name, 'trial': trial, 'learning_rate': learning_rate, 'batch_size': batch_size, 'undersample_majority': undersample, 'effective_train_count': len(trial_train), **calculate_metrics(validation['label_index'].to_numpy(), np.argmax(probabilities, axis=1))}

        all_results.append(row)

        if is_better(row, best): best = row; model.save(best_path)
        print(json.dumps(row, indent=2), flush=True)
        del model
        gc.collect()
        tf.keras.backend.clear_session()

    best['model_path'] = str(best_path)
    best_by_model[model_name] = best
    print('Best', model_name, best)

===== Training custom_cnn trial 1/2 =====
Instructions for updating:
Use `tf.data.Dataset.ignore_errors` instead.
Epoch 1/5


/home/rushd/fracturecare-ai-venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
E0000 00:00:1787800496.037244     380 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/fracatlas_fracture_classifier_1/dropout_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
I0000 00:00:1787800496.815730     416 cuda_dnn.cc:461] Loaded cuDNN version 92400


202/202 ━━━━━━━━━━━━━━━━━━━━ 36s 150ms/step - accuracy: 0.4181 - loss: 1.1079 - val_accuracy: 0.8234 - val_loss: 0.7779
Epoch 2/5
202/202 ━━━━━━━━━━━━━━━━━━━━ 30s 149ms/step - accuracy: 0.5452 - loss: 1.0378 - val_accuracy: 0.8109 - val_loss: 0.7699
Epoch 3/5
202/202 ━━━━━━━━━━━━━━━━━━━━ 30s 146ms/step - accuracy: 0.5449 - loss: 1.0116 - val_accuracy: 0.7861 - val_loss: 0.7987
Epoch 4/5
202/202 ━━━━━━━━━━━━━━━━━━━━ 29s 145ms/step - accuracy: 0.5896 - loss: 0.9835 - val_accuracy: 0.4104 - val_loss: 1.1115
{
  "model": "custom_cnn",
  "trial": 1,
  "learning_rate": 0.001,
  "batch_size": 16,
  "undersample_majority": false,
  "effective_train_count": 3219,
  "accuracy": 0.8109452736318408,
  "balanced_accuracy": 0.3334638767669986,
  "macro_precision": 0.3218806509945751,
  "macro_recall": 0.3334638767669986,
  "macro_f1": 0.30936789655120506,
  "fracture_macro_precision": 0.07142857142857142,
  "fracture_macro_recall": 0.009259259259259259,
  "fracture_macro_f1": 0.01639344262295082
}


/home/rushd/fracturecare-ai-venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


===== Training custom_cnn trial 2/2 =====
Epoch 1/5


...

{
  "model": "efficientnetb0",
  "trial": 1,
  "learning_rate": 0.001,
  "batch_size": 16,
  "undersample_majority": false,
  "effective_train_count": 3219,
  "accuracy": 0.8208955223880597,
  "balanced_accuracy": 0.6216500251213835,
  "macro_precision": 0.611672278338945,
  "macro_recall": 0.6216500251213835,
  "macro_f1": 0.6165048414165912,
  "fracture_macro_precision": 0.46296296296296297,
  "fracture_macro_recall": 0.4793028322440087,
  "fracture_macro_f1": 0.4708994708994709
}
===== Training efficientnetb0 trial 2/2 =====
Epoch 1/5


/home/rushd/fracturecare-ai-venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


113/113 ━━━━━━━━━━━━━━━━━━━━ 26s 186ms/step - accuracy: 0.5728 - loss: 0.9449 - val_accuracy: 0.7761 - val_loss: 0.5002
Epoch 2/5
113/113 ━━━━━━━━━━━━━━━━━━━━ 18s 163ms/step - accuracy: 0.6774 - loss: 0.7701 - val_accuracy: 0.7612 - val_loss: 0.5388
Epoch 3/5
113/113 ━━━━━━━━━━━━━━━━━━━━ 21s 183ms/step - accuracy: 0.7023 - loss: 0.7239 - val_accuracy: 0.7811 - val_loss: 0.5991
{
  "model": "efficientnetb0",
  "trial": 2,
  "learning_rate": 0.001,
  "batch_size": 16,
  "undersample_majority": true,
  "effective_train_count": 1807,
  "accuracy": 0.7761194029850746,
  "balanced_accuracy": 0.5528865456891047,
  "macro_precision": 0.6050326045262754,
  "macro_recall": 0.5528865456891047,
  "macro_f1": 0.567826462025089,
  "fracture_macro_precision": 0.4581818181818182,
  "fracture_macro_recall": 0.40032679738562094,
  "fracture_macro_f1": 0.4127906976744186
}


/home/rushd/fracturecare-ai-venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


Best efficientnetb0 {'model': 'efficientnetb0', 'trial': 1, 'learning_rate': 0.001, 'batch_size': 16, 'undersample_majority': False, 'effective_train_count': 3219, 'accuracy': 0.8208955223880597, 'balanced_accuracy': 0.6216500251213835, 'macro_precision': 0.611672278338945, 'macro_recall': 0.6216500251213835, 'macro_f1': 0.6165048414165912, 'fracture_macro_precision': 0.46296296296296297, 'fracture_macro_recall': 0.4793028322440087, 'fracture_macro_f1': 0.4708994708994709, 'model_path': '/mnt/c/Users/Rushd/OneDrive - wslqd/Documents/Uni Documents/ICBT/Development Project Final Year/Final Documents/fracturecare-prototype/ai-service/artifacts/tuning/efficientnetb0_best.keras'}


In [4]:
results_frame = pd.DataFrame(all_results).sort_values(['fracture_macro_recall','macro_f1','balanced_accuracy'], ascending=False).reset_index(drop=True)
best_frame = pd.DataFrame(best_by_model.values()).sort_values(['fracture_macro_recall','macro_f1','balanced_accuracy'], ascending=False).reset_index(drop=True)
display(best_frame[['model','learning_rate','batch_size','undersample_majority','fracture_macro_recall','fracture_macro_f1','macro_f1','balanced_accuracy']])

best = best_frame.iloc[0].to_dict()
selected_model = tf.keras.models.load_model(best['model_path'], compile=False)
test_probabilities = selected_model.predict(make_dataset(test, int(best['batch_size']), False), verbose=0)
test_actual = test['label_index'].to_numpy()
test_predicted = np.argmax(test_probabilities, axis=1)

final_test_metrics = calculate_metrics(test_actual, test_predicted)


print('Selected architecture:', best['model'])
print('Untouched test metrics:', json.dumps(final_test_metrics, indent=2))
print('Confusion matrix:', confusion_matrix(test_actual, test_predicted, labels=np.arange(len(CLASS_NAMES))).tolist())

,model,learning_rate,batch_size,undersample_majority,fracture_macro_recall,fracture_macro_f1,macro_f1,balanced_accuracy
0,efficientnetb0,0.001,16,False,0.479303,0.470899,0.616505,0.621650
1,mobilenetv2,0.001,16,True,0.378540,0.377305,0.545163,0.535341
2,custom_cnn,0.001,16,False,0.009259,0.016393,0.309368,0.333464


Selected architecture: efficientnetb0
Untouched test metrics: {
  "accuracy": 0.8411910669975186,
  "balanced_accuracy": 0.638362656240744,
  "macro_precision": 0.6469958126399845,
  "macro_recall": 0.638362656240744,
  "macro_f1": 0.6403776020753634,
  "fracture_macro_precision": 0.5119047619047619,
  "fracture_macro_recall": 0.5058823529411764,
  "fracture_macro_f1": 0.5054674685620557
}
Confusion matrix: [[299, 26, 6], [21, 33, 1], [6, 4, 7]]


/home/rushd/fracturecare-ai-venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


In [ ]:
final_model_path = ARTIFACT_DIR / 'fracture_classifier.keras'

selected_model.save(final_model_path)

best_frame.to_csv(ARTIFACT_DIR / 'tuned_model_comparison.csv', index=False)

def report_rows(frame, variant, source):
    if frame.empty:
        return pd.DataFrame()
    return frame.assign(evaluation_set='validation', training_variant=variant, source=source).rename(columns={'macro_precision': 'average_precision', 'macro_recall': 'average_recall', 'macro_f1': 'average_f1', 'fracture_macro_precision': 'fracture_average_precision', 'fracture_macro_recall': 'fracture_average_recall', 'fracture_macro_f1': 'fracture_average_f1'})[['evaluation_set', 'training_variant', 'model', 'accuracy', 'balanced_accuracy', 'average_precision', 'average_recall', 'average_f1', 'fracture_average_precision', 'fracture_average_recall', 'fracture_average_f1', 'source']]
untuned_path = ARTIFACT_DIR / 'model_comparison.csv'
untuned_frame = pd.read_csv(untuned_path) if untuned_path.exists() else pd.DataFrame()
final_row = pd.DataFrame([{'evaluation_set': 'test', 'training_variant': 'tuned', 'model': best['model'], **final_test_metrics, 'source': 'model_metadata.json finalTestMetrics'}]).rename(columns={'macro_precision': 'average_precision', 'macro_recall': 'average_recall', 'macro_f1': 'average_f1', 'fracture_macro_precision': 'fracture_average_precision', 'fracture_macro_recall': 'fracture_average_recall', 'fracture_macro_f1': 'fracture_average_f1'})
canonical = pd.concat([report_rows(untuned_frame, 'untuned', 'model_comparison.csv'), report_rows(best_frame, 'tuned', 'tuned_model_comparison.csv'), final_row], ignore_index=True)
#All tuned and untuned model metrics saved here
canonical.to_csv(ARTIFACT_DIR / 'FINAL_MODEL_METRICS.csv', index=False)


metadata = {'modelVersion': f'fracatlas-{best["model"]}-tuned-1.0.0', 'selectedModel': best['model'], 'architecture': best['model'], 'classes': list(CLASS_NAMES), 'imageSize': list(IMAGE_SIZE), 'dataset': 'FracAtlas', 'trainCount': int(len(train)), 'validationCount': int(len(validation)), 'testCount': int(len(test)), 'selectionMetric': SELECTION_METRIC, 'selectedHyperparameters': {key: best[key] for key in ('learning_rate','batch_size','undersample_majority','effective_train_count')}, 'comparison': best_frame.to_dict(orient='records'), 'finalTestMetrics': final_test_metrics, 'confusionMatrix': confusion_matrix(test_actual, test_predicted, labels=np.arange(len(CLASS_NAMES))).tolist()}


(ARTIFACT_DIR / 'model_metadata.json').write_text(json.dumps(metadata, indent=2, default=float), encoding='utf-8')
print('Saved final model for FastAPI:', final_model_path)